#Cell 1:

In [1]:
from google.colab import files

uploaded = files.upload()

Saving retail_store_sales.csv to retail_store_sales.csv


In [40]:
import pandas as pd

df = pd.read_csv("retail_store_sales.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (12575, 11)


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


#Cell 2 — Dataset structure check:

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


#Cell 3 — Missing values count:

In [42]:
df.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


#Cell 4 — Duplicate check:

In [43]:
df.duplicated().sum()

np.int64(0)

#Cell 5 — Missing values ka pattern:

In [44]:
id_cols = [
    "Item",
    "Price Per Unit",
    "Quantity",
    "Total Spent",
    "Discount Applied"
]

df[id_cols].isnull().value_counts()

Item   Price Per Unit  Quantity  Total Spent  Discount Applied
False  False           False     False        False               7579
                                              True                3783
True   True            False     False        False                404
       False           True      True         False                393
                                              True                 211
       True            False     False        True                 205
Name: count, dtype: int64

#Cell 6 — Price recover ho sakta hai?

In [45]:
price_recovery = df[
    df["Price Per Unit"].isna() &
    df["Quantity"].notna() &
    df["Total Spent"].notna()
]

print("Recoverable Price rows:", len(price_recovery))

Recoverable Price rows: 609


##Cell 7 — Missing prices recover karo:

In [46]:
df.loc[df["Price Per Unit"].isna(), "Price Per Unit"] = (
    df["Total Spent"] / df["Quantity"]
)

In [47]:
df["Price Per Unit"].isnull().sum()

np.int64(0)

#Cell 8 — Quantity missing values recover ho sakti hain?

In [48]:
quantity_recovery = df[
    df["Quantity"].isna() &
    df["Price Per Unit"].notna() &
    df["Total Spent"].notna()
]

print("Recoverable Quantity rows:", len(quantity_recovery))

Recoverable Quantity rows: 0


#Cell 9:

In [49]:
total_recovery = df[
    df["Total Spent"].isna() &
    df["Price Per Unit"].notna() &
    df["Quantity"].notna()
]

print("Recoverable Total Spent rows:", len(total_recovery))

Recoverable Total Spent rows: 0


# Cell-10 Item missing check:

In [50]:
df[df["Item"].isna()]["Category"].value_counts()

,count
Category,
Food,162
Computers and electric accessories,161
Patisserie,159
Milk Products,159
Electric household essentials,154
Butchers,147
Beverages,140
Furniture,131


In [51]:
item_check = df.dropna(subset=["Item", "Price Per Unit"])

item_check.groupby(["Category", "Price Per Unit"])["Item"].nunique().max()

1

#Cell 11:

In [52]:
item_recovery = df[
    df["Item"].isna() &
    df["Price Per Unit"].notna()
]

print("Recoverable Item rows:", len(item_recovery))

Recoverable Item rows: 1213


In [56]:
item_map = (
    df.dropna(subset=["Item", "Price Per Unit"])
      .drop_duplicates(["Category", "Price Per Unit"])
      .set_index(["Category", "Price Per Unit"])["Item"]
)

In [57]:
df["Item"] = df["Item"].fillna(
    df.apply(
        lambda row: item_map.get((row["Category"], row["Price Per Unit"])),
        axis=1
    )
)

In [58]:
df["Item"].isnull().sum()

np.int64(0)

In [59]:
df["Discount Applied"].value_counts(dropna=False)

,count
Discount Applied,
True,4219
NaN,4199
False,4157


In [60]:
df["Discount Applied"] = df["Discount Applied"].fillna("Unknown")

In [61]:
df.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,0
Price Per Unit,0
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


In [62]:
print("Rows before removing incomplete transactions:", len(df))

incomplete_rows = df[
    df["Quantity"].isna() &
    df["Total Spent"].isna()
]

print("Incomplete transaction rows:", len(incomplete_rows))

Rows before removing incomplete transactions: 12575
Incomplete transaction rows: 604


In [63]:
df_clean = df.dropna(
    subset=["Quantity", "Total Spent"]
).copy()

print("Clean dataset shape:", df_clean.shape)

Clean dataset shape: (11971, 11)


In [65]:
df_clean.isnull().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,0
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


In [66]:
df_clean.duplicated().sum()

np.int64(0)

In [67]:
df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"]
)

print(df_clean["Transaction Date"].dtype)

datetime64[ns]


In [68]:
df_clean[["Price Per Unit", "Quantity", "Total Spent"]].dtypes

,0
Price Per Unit,float64
Quantity,float64
Total Spent,float64


In [69]:
calculated_total = (
    df_clean["Price Per Unit"] * df_clean["Quantity"]
)

difference = (
    df_clean["Total Spent"] - calculated_total
).abs()

print("Maximum difference:", difference.max())

Maximum difference: 0.0


In [71]:
df_clean.dtypes

,0
Transaction ID,object
Customer ID,object
Category,object
Item,object
Price Per Unit,float64
Quantity,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


In [72]:
df_clean.to_csv(
    "retail_store_sales_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
